In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import GlobalAveragePooling2D
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

# ==============================================================================
# 1. 기본 설정 및 경로 정의
# ==============================================================================
# 기본 경로 설정 (사용자 환경에 맞게 수정)
# 예: './food_dataset/' 안에 'train', 'valid', 'test' 폴더가 있다고 가정
# base_dir = './food_dataset/'
# train_dir = os.path.join(base_dir, 'train')
# valid_dir = os.path.join(base_dir, 'valid')
# test_dir = os.path.join(base_dir, 'test')

train_dir = 'E:/★★★★★AI★★★★★/음식 ai data/selectStart음식DATA/Computer Vision Lab/train'
valid_dir = 'E:/★★★★★AI★★★★★/음식 ai data/selectStart음식DATA/Computer Vision Lab/valid'
test_dir = 'E:/★★★★★AI★★★★★/음식 ai data/selectStart음식DATA/Computer Vision Lab/test' # 평가용 데이터셋 경로


# 모델 파라미터 설정
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# 저장될 모델 파일 이름 정의
FEATURE_EXTRACTOR_PATH = 'food_feature_extractor.h5'
CLASSIFIER_PATH = 'food_classifier_model.json'


# ==============================================================================
# 2. ImageDataGenerator를 사용한 데이터 준비
# ==============================================================================
print("🖼️ 데이터 제너레이터를 설정합니다...")
# 훈련 데이터: 데이터 증강(Augmentation) 적용
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# 검증 및 테스트 데이터: 증강 없이 스케일 조정만 적용
test_datagen = ImageDataGenerator(rescale=1./255)

# 데이터 제너레이터 생성
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

validation_generator = test_datagen.flow_from_directory(
    valid_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# 클래스 수 및 이름 확인
num_classes = len(train_generator.class_indices)
class_labels = list(train_generator.class_indices.keys())
print(f"총 클래스 개수: {num_classes}")
print(f"클래스 이름: {class_labels}")


# ==============================================================================
# 3. 전이 학습 모델 생성 및 저장
# ==============================================================================
print("\n🤖 특징 추출기(Feature Extractor) 모델을 생성합니다...")
# include_top=False: 분류기 레이어를 제외하고 불러옴
# weights='imagenet': ImageNet으로 사전 훈련된 가중치 사용
base_model = EfficientNetV2B0(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False # 가중치 동결

feature_extractor = Model(
    inputs=base_model.input,
    outputs=GlobalAveragePooling2D()(base_model.output)
)

# ✅ 특징 추출기 모델 저장
feature_extractor.save(FEATURE_EXTRACTOR_PATH)
print(f"✅ 특징 추출기 모델이 '{FEATURE_EXTRACTOR_PATH}'에 저장되었습니다.")


# ==============================================================================
# 4. 특징 추출 함수 정의 및 실행
# ==============================================================================
def extract_features(generator, model, sample_count):
    """ImageDataGenerator로부터 특징과 레이블을 추출하는 함수"""
    features = np.zeros(shape=(sample_count, model.output_shape[1]))
    labels = np.zeros(shape=(sample_count, num_classes))
    
    i = 0
    for inputs_batch, labels_batch in generator:
        features_batch = model.predict(inputs_batch, verbose=0)
        start_index = i * BATCH_SIZE
        end_index = start_index + features_batch.shape[0]

        features[start_index:end_index] = features_batch
        labels[start_index:end_index] = labels_batch
        i += 1
        if i * BATCH_SIZE >= sample_count:
            break
            
    # 레이블을 원-핫 인코딩에서 단일 정수로 변환
    return features, np.argmax(labels, axis=1)

# 각 데이터셋에서 특징 추출
print("\n데이터셋에서 특징 추출을 시작합니다...")
train_features, train_labels = extract_features(train_generator, feature_extractor, train_generator.samples)
print("훈련 데이터 특징 추출 완료.")
val_features, val_labels = extract_features(validation_generator, feature_extractor, validation_generator.samples)
print("검증 데이터 특징 추출 완료.")
test_features, test_labels = extract_features(test_generator, feature_extractor, test_generator.samples)
print("테스트 데이터 특징 추출 완료.")

print(f"\n추출된 특징 벡터 형태 (Train): {train_features.shape}")


# ==============================================================================
# 5. XGBoost 모델 훈련, 저장 및 평가
# ==============================================================================
print("\n🚀 XGBoost 모델을 GPU로 훈련합니다...")

# XGBoost 분류기 모델 생성
xgb_classifier = xgb.XGBClassifier(
    objective='multi:softmax',    # 다중 분류 문제
    num_class=num_classes,
    eval_metric='mlogloss',
    use_label_encoder=False,
    n_estimators=500,             # 트리의 개수
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='gpu_hist'        # ❗ GPU 가속 사용
)

# 조기 종료(Early Stopping)를 사용하여 최적의 트리 개수 찾기
xgb_classifier.fit(
    train_features,
    train_labels,
    eval_set=[(val_features, val_labels)],
    early_stopping_rounds=50, # 50 라운드 동안 성능 개선이 없으면 훈련 중지
    verbose=True
)

# ✅ 훈련된 XGBoost 모델 저장
print(f"\n✅ 훈련된 XGBoost 모델이 '{CLASSIFIER_PATH}'에 저장되었습니다.")
xgb_classifier.save_model(CLASSIFIER_PATH)


# ==============================================================================
# 6. 최종 성능 평가
# ==============================================================================
print("\n📊 테스트 데이터로 최종 성능을 평가합니다.")
y_pred = xgb_classifier.predict(test_features)
accuracy = accuracy_score(test_labels, y_pred)
print(f"✅ 최종 정확도: {accuracy * 100:.2f}%")

# 분류 리포트 출력
print("\nClassification Report:")
print(classification_report(test_labels, y_pred, target_names=class_labels))

# 혼동 행렬(Confusion Matrix) 시각화
print("\n혼동 행렬(Confusion Matrix)을 시각화합니다...")
cm = confusion_matrix(test_labels, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix')
plt.ylabel('Actual Labels')
plt.xlabel('Predicted Labels')
plt.tight_layout()
plt.show()

print("\n🎉 모든 과정이 완료되었습니다.")

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
import xgboost as xgb

# 먼저 비어있는 모델을 만들고 저장된 가중치를 불러옵니다.
loaded_xgb = xgb.XGBClassifier()
loaded_xgb.load_model('food_classifier_model.json')

# 이제 이 loaded_xgb 모델로 바로 예측(predict)을 수행할 수 있습니다.
# 예: loaded_xgb.predict(새로운_데이터의_특징_벡터)